In [4]:
!pip install pandas folium numpy geopandas

import json
import math
import os
from dataclasses import dataclass, field
from typing import List, Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster

In [5]:
# 1. 利用 haversine 公式計算兩點於地球上的距離
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6371.0
    rad_lat1 = np.radians(lat1)
    rad_lon1 = np.radians(lon1)
    rad_lat2 = np.radians(lat2)
    rad_lon2 = np.radians(lon2)

    dlat = rad_lat2 - rad_lat1
    dlon = rad_lon2 - rad_lon1

    a = np.sin(dlat / 2)**2 + np.cos(rad_lat1) * np.cos(rad_lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = earth_radius * c

    return distance

# 2. 利用地震動衰減公式計算 PGA 值
def calculate_pga(magnitude, distance):
    # 定義經驗常數 (以台灣典型地震衰減模型進行簡化)
    A = 0.026
    B = 1.28
    C = 10.0  # 近震央修正項
    D = 1.5   # 幾何衰減指數

    base_pga = (A * np.exp(B * magnitude)) / ((distance + C) ** D)

    # 4. 轉換為氣象署常用的 gal 單位
    final_pga = base_pga * 980

    return final_pga

# 3. 將 PGA 轉換為震度級數
def get_intensity_level(pga):
    if pga < 0.8: return "0級"
    elif pga < 2.5: return "1級"
    elif pga < 8.0: return "2級"
    elif pga < 25.0: return "3級"
    elif pga < 80.0: return "4級"
    elif pga < 140.0: return "5弱"
    elif pga < 250.0: return "5強"
    elif pga < 440.0: return "6弱"
    elif pga < 800.0: return "6強"
    else: return "7級"

# 4. 設定老屋在該震度級數的受損機率
def get_damage_rate(intensity):
    damage_table = {
        "0級": 0.00, "1級": 0.00, "2級": 0.00, "3級": 0.00,
        "4級": 0.01,
        "5弱": 0.05,
        "5強": 0.15,
        "6弱": 0.35,
        "6強": 0.60,
        "7級": 0.85
    }
    return damage_table.get(intensity, 0.00)

# 5. 整合上方 function 進行最終計算
def run_simulation(epicenter_lat, epicenter_lon, magnitude, area_df):
    results = []
    for idx, row in area_df.iterrows():
        dist = haversine_distance(epicenter_lat, epicenter_lon, row['lat'], row['lon'])
        pga = calculate_pga(magnitude, dist)
        intensity = get_intensity_level(pga)
        damage_rate = get_damage_rate(intensity)

        # 災民預估模型公式： 該里總人口 * 該里老屋比例 * 該震度下的老屋損壞率 * 0.8
        predicted_refugees = int(row['population'] * row['old house ratio'] * damage_rate * 0.8)

        results.append({
            '行政區': row['name1'],
            '里名': row['name2'],
            'lat': row['lat'],
            'lon': row['lon'],
            '震央距離_km': round(dist, 2),
            '預估PGA': round(pga, 2),
            '預估震度': intensity,
            '預估避難人數': predicted_refugees
        })

    return pd.DataFrame(results)


if __name__ == "__main__":

    csv_filename = "village.csv" #檔名可修改，需與程式碼放在同一資料夾

    # 1.從外部 CSV 檔案讀取資料
    try:
        df_areas = pd.read_csv(csv_filename, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"錯誤：找不到檔案 '{csv_filename}'，請確認檔案路徑是否正確。")
        exit()

    # 2.輸入
    try:
        epi_lat = float(input("請輸入震央緯度（例如 24.15）："))
        epi_lon = float(input("請輸入震央經度（例如 121.62）："))
        mag = float(input("請輸入地震規模（例如 6.0）："))
    except ValueError:
        print("錯誤：輸入格式不正確，經緯度與規模必須是數字。")
        exit()

    # 3.執行模擬
    df_output = run_simulation(epi_lat, epi_lon, mag, df_areas)

    # 4.輸出
    json_result = df_output.to_json(orient='records', force_ascii=False, indent=4)
    print(json_result)

    # 5.匯出成實體 .json 檔案
    output_json_file = "earthquake_simulation_result.json"
    with open(output_json_file, "w", encoding="utf-8") as f:
        f.write(json_result)

    print(f"\n[系統提示] JSON 檔案已成功匯出至: {output_json_file}")

請輸入震央緯度（例如 24.15）：24.15
請輸入震央經度（例如 121.62）：121.62
請輸入地震規模（例如 6.0）：6.0
[
    {
        "行政區":"中山區",
        "里名":"下埤里",
        "lat":25.0627,
        "lon":121.5304,
        "震央距離_km":101.89,
        "預估PGA":46.6,
        "預估震度":"4級",
        "預估避難人數":5345
    },
    {
        "行政區":"中山區",
        "里名":"中原里",
        "lat":25.0579,
        "lon":121.539,
        "震央距離_km":101.29,
        "預估PGA":46.98,
        "預估震度":"4級",
        "預估避難人數":3407
    },
    {
        "行政區":"中山區",
        "里名":"中吉里",
        "lat":25.0671,
        "lon":121.5376,
        "震央距離_km":102.32,
        "預估PGA":46.34,
        "預估震度":"4級",
        "預估避難人數":3728
    },
    {
        "行政區":"中山區",
        "里名":"中央里",
        "lat":25.0601,
        "lon":121.533,
        "震央距離_km":101.58,
        "預估PGA":46.8,
        "預估震度":"4級",
        "預估避難人數":2625
    },
    {
        "行政區":"中山區",
        "里名":"中山里",
        "lat":25.0581,
        "lon":121.5278,
        "震央距離_km":101.41,
        "預估PGA":46.91,
        "預估震度":"4

In [7]:
# 上傳到 Colab 後，通常只需要改這裡的檔名
SHELTER_CSV = "shelter.csv"
VILLAGE_CSV = "village.csv"
IMPACT_JSON = "earthquake_simulation_result.json"

# Shapefile 需要 .shp/.shx/.dbf/.prj/.cpg 一起上傳
VILLAGE_SHP = "village_boundary.shp"

def first_existing_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到檔案，請確認已上傳：{candidates}")

print("避難所 CSV：", SHELTER_CSV)
print("村里 CSV：", VILLAGE_CSV)
print("地震模擬 JSON：", IMPACT_JSON)
print("里界 Shapefile：", VILLAGE_SHP)

避難所 CSV： shelter.csv
村里 CSV： village.csv
地震模擬 JSON： earthquake_simulation_result.json
里界 Shapefile： village_boundary.shp


In [8]:
@dataclass
class Shelter:
    id: str
    name: str
    lat: float
    lon: float
    capacity: int
    city: str = ""
    district: str = ""
    village: str = ""
    address: str = ""
    allocated: int = field(default=0, repr=False)

    @property
    def remaining(self) -> int:
        return self.capacity - self.allocated

    @property
    def utilisation(self) -> float:
        if self.capacity == 0:
            return float("inf")
        return self.allocated / self.capacity


@dataclass
class Flow:
    from_zone: str
    to_shelter: str
    to_shelter_id: str
    people: int
    overflow: bool = False

    def to_dict(self) -> dict:
        return {
            "from_zone": self.from_zone,
            "to_shelter": self.to_shelter,
            "to_shelter_id": self.to_shelter_id,
            "people": self.people,
            "overflow": self.overflow,
        }


def normalize_name(text):
    return (
        str(text)
        .replace("/", "")
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("臺", "台")
        .replace("蔀", "廍")
        .strip()
    )


def fix_mojibake(text):
    """嘗試修正 shapefile 欄位中文亂碼；若沒有亂碼則原樣回傳。"""
    try:
        return str(text).encode("latin1").decode("utf-8")
    except Exception:
        return str(text)


def haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in kilometres."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def get_first_existing_column(df, candidates, required=True):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f"找不到欄位，候選欄位為：{candidates}\n目前欄位：{list(df.columns)}")
    return None


def coalesce_column(df, candidates, default=""):
    """從多個候選欄位中取第一個存在的欄位；若都不存在則回傳 default。"""
    col = get_first_existing_column(df, candidates, required=False)
    if col is None:
        return pd.Series([default] * len(df), index=df.index)
    return df[col]


def ensure_impact_columns(df: pd.DataFrame) -> pd.DataFrame:
    """把不同來源的受災資料欄位統一成演算法需要的欄位。"""
    df = df.copy()
    rename_map = {}

    if "行政區" not in df.columns:
        for c in ["name1", "district", "TOWNNAME", "行政區名"]:
            if c in df.columns:
                rename_map[c] = "行政區"
                break

    if "里名" not in df.columns:
        for c in ["name2", "village", "VILLNAME", "村里名"]:
            if c in df.columns:
                rename_map[c] = "里名"
                break

    if "預估避難人數" not in df.columns:
        for c in ["affected_population", "refugees", "evacuees", "避難人數"]:
            if c in df.columns:
                rename_map[c] = "預估避難人數"
                break

    df = df.rename(columns=rename_map)

    required = ["行政區", "里名", "lat", "lon", "預估避難人數"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"受災資料缺少必要欄位：{missing}\n目前欄位：{list(df.columns)}")

    df["lat"] = df["lat"].astype(float)
    df["lon"] = df["lon"].astype(float)
    df["預估避難人數"] = pd.to_numeric(df["預估避難人數"], errors="coerce").fillna(0).astype(int)
    return df

In [9]:
# 讀取原始 DataFrame，後面地圖與表格會用到
shelter_df = pd.read_csv(SHELTER_CSV, encoding="utf-8-sig", dtype=str)
village_df = pd.read_csv(VILLAGE_CSV, encoding="utf-8-sig", dtype=str)
gdf = gpd.read_file(VILLAGE_SHP)

# 清理欄位名稱
shelter_df.columns = [c.strip() for c in shelter_df.columns]
village_df.columns = [c.strip() for c in village_df.columns]
gdf.columns = [c.strip() for c in gdf.columns]

# 數值欄位轉型
shelter_df["lat"] = shelter_df["lat"].astype(float)
shelter_df["lon"] = shelter_df["lon"].astype(float)
shelter_df["capacity"] = pd.to_numeric(shelter_df["capacity"], errors="coerce")

skipped_shelters = shelter_df[shelter_df["capacity"].isna()].copy()
if len(skipped_shelters) > 0:
    print("略過 capacity 非數字的避難所：")

shelter_df = shelter_df.dropna(subset=["capacity", "lat", "lon"]).copy()
shelter_df["capacity"] = shelter_df["capacity"].astype(int)

village_df["lat"] = village_df["lat"].astype(float)
village_df["lon"] = village_df["lon"].astype(float)
village_df["population"] = pd.to_numeric(village_df["population"], errors="coerce").fillna(0).astype(int)

print("shelter_df columns:", list(shelter_df.columns))#可刪
print("village_df columns:", list(village_df.columns))#可刪
print("gdf columns:", list(gdf.columns))#可刪
print()
print("避難所數量：", len(shelter_df))#可刪
print("村里數量：", len(village_df))#可刪
print("里界數量：", len(gdf))#可刪

略過 capacity 非數字的避難所：


,id,name,capacity
120,SA105-1029,臺北市立圖書館松山分館(暫停用),NaN


shelter_df columns: ['id', 'name', 'capacity', 'lat', 'lon', 'address1', 'address2', 'address3', 'address4']
village_df columns: ['id1', 'id2', 'name1', 'name2', 'lat', 'lon', 'population', 'address(office)', 'old house ratio']
gdf columns: ['AREA', 'NEW', 'FULL', 'PERF_ID', 'COUN_ID', 'CPID', 'CPTID', 'CPTVID', 'NPID', 'NPTID', 'NPTVID', 'PNAME', 'TNAME', 'VNAME', 'PTVNAME', 'PTNAME', 'TVNAME', 'TM2_MAX_X', 'TM2_MAX_Y', 'TM2_MIN_X', 'TM2_MIN_Y', 'MAX_X', 'MAX_Y', 'MIN_X', 'MIN_Y', 'SECT_NAME', 'LIE_NAME', 'sn', 'geometry']

避難所數量： 418
村里數量： 456
里界數量： 456


In [10]:
def load_shelters_from_df(df: pd.DataFrame) -> List[Shelter]:
    """
    依 shelter_df 欄位建立 Shelter 物件。
    可支援：
    - 新版欄位：id, name, capacity, lat, lon, city, district, village, address
    - 目前欄位：id, name, capacity, lat, lon, address1, address2, address3, address4
    """
    shelters = []

    for _, row in df.iterrows():
        address_parts = [
            str(row.get(col, "")).strip()
            for col in ["address1", "address2", "address3", "address4", "address"]
            if col in df.columns and pd.notna(row.get(col, ""))
        ]
        address = "".join(address_parts)

        shelters.append(
            Shelter(
                id=str(row["id"]).strip(),
                name=str(row["name"]).strip(),
                lat=float(row["lat"]),
                lon=float(row["lon"]),
                capacity=int(row["capacity"]),
                city=str(row.get("city", row.get("address1", ""))).strip(),
                district=str(row.get("district", row.get("address2", ""))).strip(),
                village=str(row.get("village", row.get("address3", ""))).strip(),
                address=address,
            )
        )

    return shelters


shelters = load_shelters_from_df(shelter_df)

total_cap = sum(s.capacity for s in shelters)
print(f"Shelters loaded : {len(shelters)}   total capacity = {total_cap:,}")#可刪
display(shelter_df.head())#可刪

Shelters loaded : 418   total capacity = 488,210


,id,name,capacity,lat,lon,address1,address2,address3,address4
0,SA100-0002,臺北市立螢橋國民中學,267,25.016335,121.530379,臺北市,中正區,林興里,汀州路三段四號
1,SA100-0003,臺北市立大學附設實驗國民小學,973,25.040125,121.516422,臺北市,中正區,黎明里,公園路29號
2,SA100-0004,臺北市立弘道國民中學,733,25.041697,121.516246,臺北市,中正區,建國里,公園路21號
3,SA100-0005,二二八和平公園,5424,25.041076,121.514937,臺北市,中正區,黎明里,凱達格蘭大道3號
4,SA100-0006,臺北市中正運動中心,483,25.038166,121.521832,臺北市,中正區,東門里,信義路一段1號


In [11]:
# earthquake_simulation_result.json 預期格式：
# [
#   {
#     "行政區": "信義區",
#     "里名": "興雅里",
#     "lat": 25.0423,
#     "lon": 121.5666,
#     "預估避難人數": 896,
#     "預估PGA": 0.88,         # 可選
#     "預估震度": 6,           # 可選
#     "震央距離_km": 3.1       # 可選
#   },
#   ...
# ]

with open(IMPACT_JSON, "r", encoding="utf-8") as f:
    raw_impact_data = json.load(f)

affected_df = ensure_impact_columns(pd.DataFrame(raw_impact_data))
impact_data = affected_df.to_dict(orient="records")

total_refugees = sum(int(z["預估避難人數"]) for z in impact_data)

print(f"Impact zones   : {len(impact_data)}")
print(f"Total refugees : {total_refugees:,}")
print(f"Total capacity : {sum(s.capacity for s in shelters):,}")

# 若沒有 risk_score，優先用 預估PGA；若也沒有，就用預估避難人數做 0~1 標準化
if "risk_score" not in affected_df.columns:
    if "預估PGA" in affected_df.columns:
        affected_df["risk_score"] = pd.to_numeric(affected_df["預估PGA"], errors="coerce").fillna(0)
    else:
        max_people = affected_df["預估避難人數"].max()
        affected_df["risk_score"] = affected_df["預估避難人數"] / max_people if max_people > 0 else 0

affected_df["affected_population"] = affected_df["預估避難人數"]
affected_df["merge_name"] = (
    affected_df["行政區"].astype(str) + affected_df["里名"].astype(str)
).apply(normalize_name)
affected_df["village_only_name"] = affected_df["里名"].astype(str).apply(normalize_name)

display(affected_df.head())

Impact zones   : 456
Total refugees : 1,286,830
Total capacity : 488,210


,行政區,里名,lat,lon,震央距離_km,預估PGA,預估震度,預估避難人數,risk_score,affected_population,merge_name,village_only_name
0,中山區,下埤里,25.0627,121.5304,101.89,46.60,4級,5345,46.60,5345,中山區下埤里,下埤里
1,中山區,中原里,25.0579,121.5390,101.29,46.98,4級,3407,46.98,3407,中山區中原里,中原里
2,中山區,中吉里,25.0671,121.5376,102.32,46.34,4級,3728,46.34,3728,中山區中吉里,中吉里
3,中山區,中央里,25.0601,121.5330,101.58,46.80,4級,2625,46.80,2625,中山區中央里,中央里
4,中山區,中山里,25.0581,121.5278,101.41,46.91,4級,4375,46.91,4375,中山區中山里,中山里


In [12]:
def greedy_allocate(
    impact_data: List[dict],
    shelters: List[Shelter],
) -> List[Flow]:
    """
    Distribute refugees to shelters using a greedy nearest-first strategy.

    impact_data keys required: 行政區, 里名, lat, lon, 預估避難人數
    Shelter.allocated is updated in-place.
    """
    flows: List[Flow] = []

    # 避難人數較多的里先分配，避免大群體最後只能被分到太遠的避難所
    sorted_zones = sorted(
        impact_data,
        key=lambda z: int(z["預估避難人數"]),
        reverse=True
    )

    for zone in sorted_zones:
        remaining = int(zone["預估避難人數"])
        if remaining <= 0:
            continue

        label = f"{zone['行政區']}-{zone['里名']}"

        by_dist = sorted(
            shelters,
            key=lambda s: haversine(
                float(zone["lat"]),
                float(zone["lon"]),
                s.lat,
                s.lon
            )
        )

        for shelter in by_dist:
            if remaining <= 0:
                break

            avail = shelter.remaining
            if avail <= 0:
                continue

            send = min(remaining, avail)
            shelter.allocated += send
            remaining -= send

            flows.append(
                Flow(
                    from_zone=label,
                    to_shelter=shelter.name,
                    to_shelter_id=shelter.id,
                    people=send,
                    overflow=False,
                )
            )

        # Soft overflow：全部避難所都滿了，剩餘人口仍分給使用率最低者，並標示 overflow
        if remaining > 0:
            fallback = min(by_dist, key=lambda s: s.utilisation)
            fallback.allocated += remaining

            flows.append(
                Flow(
                    from_zone=label,
                    to_shelter=fallback.name,
                    to_shelter_id=fallback.id,
                    people=remaining,
                    overflow=True,
                )
            )

    return flows


# Reset allocations in case you re-run this cell
for s in shelters:
    s.allocated = 0

flows = greedy_allocate(impact_data, shelters)

normal_flows = [f for f in flows if not f.overflow]
overflow_flows = [f for f in flows if f.overflow]
total_alloc = sum(f.people for f in flows)
total_overflow = sum(f.people for f in overflow_flows)

print(f"Total flows     : {len(flows)}")
print(f"Normal          : {len(normal_flows)} → {sum(f.people for f in normal_flows):,} people")
print(f"Overflow ⚠️     : {len(overflow_flows)} → {total_overflow:,} people")
print(f"Total allocated : {total_alloc:,} / {total_refugees:,}")

if overflow_flows:
    print("\n⚠️ Overflow detail:")
    for f in overflow_flows:
        print(f"  {f.from_zone} → {f.to_shelter} (+{f.people:,} over capacity)")

Total flows     : 865
Normal          : 521 → 488,210 people
Overflow ⚠️     : 344 → 798,620 people
Total allocated : 1,286,830 / 1,286,830

⚠️ Overflow detail:
  大安區-德安里 → 臺北市信義運動中心 (+2,709 over capacity)
  中山區-力行里 → 明水公園 (+3,858 over capacity)
  文山區-興福里 → 臺北市文山區溪口國民小學 (+3,856 over capacity)
  大安區-仁慈里 → 臺北市大安區公館國民小學 (+3,841 over capacity)
  中正區-文北里 → 臺北市立中正國民中學 (+3,834 over capacity)
  大安區-芳和里 → 國立臺灣師範大學 (+3,801 over capacity)
  中山區-興亞里 → 臺北市中山區吉林國民小學 (+3,775 over capacity)
  松山區-富錦里 → 臺北市市場處龍城市場地下停車場 (+3,775 over capacity)
  文山區-華興里 → 臺北市市場處興隆市場地下停車場 (+3,771 over capacity)
  北投區-溫泉里 → 臺北市北投區北投國民小學 (+3,745 over capacity)
  松山區-莊敬里 → 臺北市中山區長安國民小學 (+3,743 over capacity)
  士林區-後港里 → 臺北市士林區葫蘆國民小學 (+3,740 over capacity)
  萬華區-和德里 → 臺北市立萬華國民中學 (+3,740 over capacity)
  北投區-永欣里 → 臺北市立圖書館吉利分館 (+3,732 over capacity)
  中山區-中吉里 → 金泰(公8)公園 (+3,728 over capacity)
  大同區-建明里 → 臺北市大同區大橋國民小學 (+3,728 over capacity)
  大安區-福住里 → 大安站 (+3,718 over capacity)
  中山區-朱馥里 → 臺北市立圖書館長安分館 (+3,711 over capacity)
  大

In [13]:
flows_df = pd.DataFrame([f.to_dict() for f in flows])

# 拆出行政區、里名，方便跟地圖資料合併
flows_df[["行政區", "里名"]] = flows_df["from_zone"].str.split("-", n=1, expand=True)
flows_df["merge_name"] = (flows_df["行政區"] + flows_df["里名"]).apply(normalize_name)
flows_df["overflow_text"] = flows_df["overflow"].map({True: "⚠️", False: ""})

display(flows_df.head())

,from_zone,to_shelter,to_shelter_id,people,overflow,行政區,里名,merge_name,overflow_text
0,大安區-龍淵里,臺北市大安區仁愛國民小學,SA106-0009,1755,False,大安區,龍淵里,大安區龍淵里,
1,大安區-龍淵里,大安森林公園,SA106-0005,5360,False,大安區,龍淵里,大安區龍淵里,
2,松山區-復盛里,民生公園,SA105-1032,1554,False,松山區,復盛里,松山區復盛里,
3,松山區-復盛里,敦化國小,SA105-1017,922,False,松山區,復盛里,松山區復盛里,
4,松山區-復盛里,臺北市立圖書館中崙分館,SA105-1027,37,False,松山區,復盛里,松山區復盛里,


In [14]:
# 每個避難所的使用狀況
shelter_summary = pd.DataFrame([
    {
        "id": s.id,
        "name": s.name,
        "district": s.district,
        "village": s.village,
        "capacity": s.capacity,
        "allocated": s.allocated,
        "remaining": s.capacity - min(s.allocated, s.capacity),
        "over_cap": max(0, s.allocated - s.capacity),
        "util%": round(s.utilisation * 100, 1),
        "lat": s.lat,
        "lon": s.lon,
        "address": s.address,
    }
    for s in shelters
    if s.allocated > 0
]).sort_values("allocated", ascending=False).reset_index(drop=True)

display(shelter_summary)

,id,name,district,village,capacity,allocated,remaining,over_cap,util%,lat,lon,address
0,SA106-0005,大安森林公園,大安區,龍門里,37949,41413,0,3464,109.1,25.01965,121.55175,臺北市大安區龍門里新生南路二段1號
1,SA108-0004,青年公園,萬華區,騰雲里,31449,33056,0,1607,105.1,25.02840,121.49975,臺北市萬華區騰雲里水源路199號
2,SA104-0045,花博公園\n圓山園區,中山區,圓山里,13517,16690,0,3173,123.5,25.06465,121.52575,"臺北市中山區圓山里玉門街1號.(玉泉街以東,中山北路以西,酒泉街以北)"
3,SA108-0025,臺北市市場處環南中繼市場停車場,萬華區,綠堤里,7951,10778,0,2827,135.6,25.04290,121.50275,臺北市萬華區綠堤里環河南路二段338號4樓-5樓
4,SA106-0038,臺北市和平實驗國民小學,大安區,臥龍里,7418,10295,0,2877,138.8,25.02740,121.54725,臺北市大安區臥龍里敦南街76巷28號
...,...,...,...,...,...,...,...,...,...,...,...,...
413,SA103-0024,臺北市立圖書館延平分館,大同區,延平里,73,73,0,0,100.0,25.07065,121.52200,臺北市大同區延平里保安街47號
414,SA105-1022,慈祐區民活動中心,松山區,慈祐里,54,54,0,0,100.0,25.06540,121.54575,臺北市松山區慈祐里八德路4段568號
415,SA115-0009,臺北市南港區玉成國民小學,南港區,西新里,37,37,0,0,100.0,25.04590,121.60775,臺北市南港區西新里向陽路31號
416,SA114-0031,港華區民活動中心,內湖區,港華里,36,36,0,0,100.0,25.06740,121.59575,臺北市內湖區港華里環山路二段68巷14號


In [15]:
# 地圖標記用：合併 shelter 座標與分配資料
assigned_shelter_df = flows_df.merge(
    shelter_df,
    left_on="to_shelter_id",
    right_on="id",
    how="left",
    suffixes=("_flow", "_shelter")
)

assigned_shelter_df = assigned_shelter_df.dropna(subset=["lat", "lon"]).copy()

print("分配明細筆數：", len(assigned_shelter_df))
print("實際使用避難所數：", assigned_shelter_df["to_shelter_id"].nunique())
display(assigned_shelter_df.head())

分配明細筆數： 855
實際使用避難所數： 413


,from_zone,to_shelter,to_shelter_id,people,overflow,行政區,里名,merge_name,overflow_text,id,name,capacity,lat,lon,address1,address2,address3,address4
0,大安區-龍淵里,臺北市大安區仁愛國民小學,SA106-0009,1755,False,大安區,龍淵里,大安區龍淵里,,SA106-0009,臺北市大安區仁愛國民小學,1755.0,25.01840,121.55150,臺北市,大安區,敦安里,安和路一段60號
1,大安區-龍淵里,大安森林公園,SA106-0005,5360,False,大安區,龍淵里,大安區龍淵里,,SA106-0005,大安森林公園,37949.0,25.01965,121.55175,臺北市,大安區,龍門里,新生南路二段1號
2,松山區-復盛里,民生公園,SA105-1032,1554,False,松山區,復盛里,松山區復盛里,,SA105-1032,民生公園,1554.0,25.05165,121.56425,臺北市,松山區,介壽里,民生東路五段36巷4弄與8弄間.
3,松山區-復盛里,敦化國小,SA105-1017,922,False,松山區,復盛里,松山區復盛里,,SA105-1017,敦化國小,922.0,25.05140,121.56075,臺北市,松山區,中正里,敦化北路2號
4,松山區-復盛里,臺北市立圖書館中崙分館,SA105-1027,37,False,松山區,復盛里,松山區復盛里,,SA105-1027,臺北市立圖書館中崙分館,37.0,25.05640,121.56075,臺北市,松山區,中正里,長安東路二段229號7-10樓


In [16]:
# 依照 shapefile 欄位自動找出行政區與里名欄位
shp_district_col = get_first_existing_column(
    gdf,
    ["TOWNNAME", "TOWN", "行政區", "行政區名", "name1", "district"],
    required=False
)

shp_village_col = get_first_existing_column(
    gdf,
    ["VILLNAME", "TVNAME", "村里名", "里名", "name2", "village", "FULL", "name", "Name"],
    required=True
)

gdf["shp_village_fix"] = gdf[shp_village_col].apply(fix_mojibake)

if shp_district_col is not None:
    gdf["shp_district_fix"] = gdf[shp_district_col].apply(fix_mojibake)
    gdf["merge_name"] = (gdf["shp_district_fix"] + gdf["shp_village_fix"]).apply(normalize_name)
else:
    gdf["merge_name"] = gdf["shp_village_fix"].apply(normalize_name)

# 第一次：用「行政區＋里名」合併
map_df = gdf.merge(
    affected_df[["merge_name", "risk_score", "affected_population", "行政區", "里名"]],
    on="merge_name",
    how="left"
)

# 若合併不到，改用「里名」合併一次
matched_count = (map_df["affected_population"].fillna(0) > 0).sum()
if matched_count == 0:
    print("提醒：用「行政區＋里名」沒有合併到資料，改用「里名」嘗試合併。")
    gdf["village_only_name"] = gdf["shp_village_fix"].apply(normalize_name)

    map_df = gdf.merge(
        affected_df[["village_only_name", "risk_score", "affected_population", "行政區", "里名"]],
        on="village_only_name",
        how="left"
    )

map_df["risk_score"] = map_df["risk_score"].fillna(0)
map_df["affected_population"] = map_df["affected_population"].fillna(0).astype(int)

print("使用的 shapefile 行政區欄位：", shp_district_col)
print("使用的 shapefile 里名欄位：", shp_village_col)
print("成功合併受災資料的里數：", (map_df["affected_population"] > 0).sum())
display(map_df[[c for c in ["merge_name", "village_only_name", "risk_score", "affected_population"] if c in map_df.columns]].head())

使用的 shapefile 行政區欄位： None
使用的 shapefile 里名欄位： TVNAME
成功合併受災資料的里數： 448


,merge_name,risk_score,affected_population
0,中正區龍福里,49.20,3434
1,萬華區福星里,48.56,254
2,大安區龍坡里,50.04,3464
3,萬華區新起里,48.43,2850
4,中正區建國里,48.82,1136


In [17]:
def risk_color(risk):
    if risk >= 0.8:
        return "#ff0000"
    elif risk >= 0.5:
        return "#ff8800"
    elif risk > 0:
        return "#ffff00"
    else:
        return "#cccccc"


m = folium.Map(
    location=[25.04, 121.56],
    zoom_start=11
)

folium.GeoJson(
    map_df,
    style_function=lambda feature: {
        "fillColor": risk_color(feature["properties"].get("risk_score", 0)),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.9 if feature["properties"].get("risk_score", 0) > 0 else 0.1,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[c for c in ["行政區", "里名", "risk_score", "affected_population"] if c in map_df.columns],
        aliases=["行政區", "里名", "風險值", "預估避難人數"],
        localize=True
    )
).add_to(m)

marker_cluster = MarkerCluster(name="已分配避難所").add_to(m)

for _, row in assigned_shelter_df.iterrows():
    overflow_mark = "⚠️ 超量分配<br>" if bool(row["overflow"]) else ""

    folium.Marker(
        location=[float(row["lat"]), float(row["lon"])],
        popup=f"""
        {overflow_mark}
        避難所：{row['to_shelter']}<br>
        容量：{row['capacity']}<br>
        本筆分配人數：{row['people']}<br>
        來源受災區域：{row['行政區']}{row['里名']}
        """,
        tooltip=f"{row['to_shelter']}｜分配 {row['people']} 人",
        icon=folium.Icon(
            color="red" if bool(row["overflow"]) else "blue",
            icon="home"
        )
    ).add_to(marker_cluster)

folium.LayerControl().add_to(m)

m

In [18]:
# 受災里統整表
affected_table = affected_df[[
    "行政區",
    "里名",
    "risk_score",
    "affected_population",
    "lat",
    "lon"
]].copy()

affected_table = affected_table.rename(columns={
    "risk_score": "風險值",
    "affected_population": "預估避難人數",
    "lat": "緯度",
    "lon": "經度"
}).sort_values("預估避難人數", ascending=False)

display(affected_table)

,行政區,里名,風險值,預估避難人數,緯度,經度
339,大安區,龍淵里,50.02,7115,25.0179,121.5538
400,松山區,復盛里,47.40,6488,25.0535,121.5630
44,中正區,南福里,48.13,6437,25.0405,121.5226
189,北投區,立農里,42.05,6141,25.1305,121.4888
401,松山區,慈祐里,46.55,5830,25.0651,121.5546
...,...,...,...,...,...,...
213,南港區,重陽里,48.10,0,25.0453,121.6150
117,內湖區,南湖里,46.97,0,25.0607,121.5926
74,信義區,三犁里,48.99,0,25.0325,121.5788
432,萬華區,新和里,48.41,0,25.0345,121.5014


In [19]:
# 避難所分配明細表
allocation_table = assigned_shelter_df[[
    "行政區",
    "里名",
    "to_shelter",
    "to_shelter_id",
    "people",
    "capacity",
    "lat",
    "lon",
    "overflow_text"
]].copy()

allocation_table = allocation_table.rename(columns={
    "行政區": "來源行政區",
    "里名": "來源里名",
    "to_shelter": "避難所名稱",
    "to_shelter_id": "避難所ID",
    "people": "分配人數",
    "capacity": "避難所容量",
    "lat": "避難所緯度",
    "lon": "避難所經度",
    "overflow_text": "是否超量"
})

display(allocation_table)

,來源行政區,來源里名,避難所名稱,避難所ID,分配人數,避難所容量,避難所緯度,避難所經度,是否超量
0,大安區,龍淵里,臺北市大安區仁愛國民小學,SA106-0009,1755,1755.0,25.018400,121.551500,
1,大安區,龍淵里,大安森林公園,SA106-0005,5360,37949.0,25.019650,121.551750,
2,松山區,復盛里,民生公園,SA105-1032,1554,1554.0,25.051650,121.564250,
3,松山區,復盛里,敦化國小,SA105-1017,922,922.0,25.051400,121.560750,
4,松山區,復盛里,臺北市立圖書館中崙分館,SA105-1027,37,37.0,25.056400,121.560750,
...,...,...,...,...,...,...,...,...,...
860,信義區,黎順里,臺北市文山區興華國民小學,SA116-0028,92,241.0,24.998432,121.554101,⚠️
861,萬華區,騰雲里,臺北市立螢橋國民中學,SA100-0002,81,267.0,25.016335,121.530379,⚠️
862,信義區,黎忠里,臺北市立育成高級中學,SA115-0002,73,5631.0,25.051150,121.602000,⚠️
863,信義區,富台里,臺北市大安區幸安國民小學,SA106-0004,43,640.0,25.017150,121.531750,⚠️


In [20]:
# 避難所總使用表
shelter_use_table = shelter_summary.rename(columns={
    "id": "避難所ID",
    "name": "避難所名稱",
    "district": "行政區",
    "village": "里名",
    "capacity": "容量",
    "allocated": "總分配人數",
    "remaining": "剩餘容量",
    "over_cap": "超量人數",
    "util%": "使用率%",
    "lat": "緯度",
    "lon": "經度",
    "address": "地址"
})

display(shelter_use_table)

,避難所ID,避難所名稱,行政區,里名,容量,總分配人數,剩餘容量,超量人數,使用率%,緯度,經度,地址
0,SA106-0005,大安森林公園,大安區,龍門里,37949,41413,0,3464,109.1,25.01965,121.55175,臺北市大安區龍門里新生南路二段1號
1,SA108-0004,青年公園,萬華區,騰雲里,31449,33056,0,1607,105.1,25.02840,121.49975,臺北市萬華區騰雲里水源路199號
2,SA104-0045,花博公園\n圓山園區,中山區,圓山里,13517,16690,0,3173,123.5,25.06465,121.52575,"臺北市中山區圓山里玉門街1號.(玉泉街以東,中山北路以西,酒泉街以北)"
3,SA108-0025,臺北市市場處環南中繼市場停車場,萬華區,綠堤里,7951,10778,0,2827,135.6,25.04290,121.50275,臺北市萬華區綠堤里環河南路二段338號4樓-5樓
4,SA106-0038,臺北市和平實驗國民小學,大安區,臥龍里,7418,10295,0,2877,138.8,25.02740,121.54725,臺北市大安區臥龍里敦南街76巷28號
...,...,...,...,...,...,...,...,...,...,...,...,...
413,SA103-0024,臺北市立圖書館延平分館,大同區,延平里,73,73,0,0,100.0,25.07065,121.52200,臺北市大同區延平里保安街47號
414,SA105-1022,慈祐區民活動中心,松山區,慈祐里,54,54,0,0,100.0,25.06540,121.54575,臺北市松山區慈祐里八德路4段568號
415,SA115-0009,臺北市南港區玉成國民小學,南港區,西新里,37,37,0,0,100.0,25.04590,121.60775,臺北市南港區西新里向陽路31號
416,SA114-0031,港華區民活動中心,內湖區,港華里,36,36,0,0,100.0,25.06740,121.59575,臺北市內湖區港華里環山路二段68巷14號


In [21]:
shelter_summary = pd.DataFrame([
    {
        'id':        s.id,
        'name':      s.name,
        'district':  s.district,
        'village':   s.village,
        'capacity':  s.capacity,
        'allocated': s.allocated,
        'over_cap':  max(0, s.allocated - s.capacity),
        'util%':     round(s.utilisation * 100, 1),
    }
    for s in shelters
    if s.allocated > 0
]).sort_values('allocated', ascending=False).reset_index(drop=True)

display(shelter_summary)

,id,name,district,village,capacity,allocated,over_cap,util%
0,SA106-0005,大安森林公園,大安區,龍門里,37949,41413,3464,109.1
1,SA108-0004,青年公園,萬華區,騰雲里,31449,33056,1607,105.1
2,SA104-0045,花博公園\n圓山園區,中山區,圓山里,13517,16690,3173,123.5
3,SA108-0025,臺北市市場處環南中繼市場停車場,萬華區,綠堤里,7951,10778,2827,135.6
4,SA106-0038,臺北市和平實驗國民小學,大安區,臥龍里,7418,10295,2877,138.8
...,...,...,...,...,...,...,...,...
413,SA103-0024,臺北市立圖書館延平分館,大同區,延平里,73,73,0,100.0
414,SA105-1022,慈祐區民活動中心,松山區,慈祐里,54,54,0,100.0
415,SA115-0009,臺北市南港區玉成國民小學,南港區,西新里,37,37,0,100.0
416,SA114-0031,港華區民活動中心,內湖區,港華里,36,36,0,100.0
